In [1]:
%load_ext Cython

In [2]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True
import cython

cdef inline double f(double x) noexcept:
    # Пример интегрируемой функции: f(x) = x^2 + 2x + 1
    return x * x + 2.0 * x + 1.0

@cython.boundscheck(False)
@cython.wraparound(False)
def integrate_trapezoidal(double a, double b, int n):
    """
    Вычисляет определенный интеграл функции f(x) на отрезке [a, b] 
    методом трапеций с разбиением на n отрезков.
    """
    if n <= 0:
        raise ValueError("Количество разбиений n должно быть больше нуля.")
        
    cdef double h = (b - a) / n  # Шаг интегрирования
    cdef double total_sum = 0.5 * (f(a) + f(b))
    cdef double x
    cdef int i
    
    # Сверхбыстрый C-цикл для суммирования внутренних точек
    for i in range(1, n):
        x = a + i * h
        total_sum += f(x)
        
    return total_sum * h

вычисление площади функции при интегрировании методом трапеций

In [3]:
%timeit integrate_trapezoidal(0.0, 10.0, 1000000)

924 μs ± 3.38 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [4]:
def f(x):
    # f(x) = x^2 + 2x + 1
    return x * x + 2.0 * x + 1.0

def integrate_trapezoidal_python(a, b, n):
    """
    Вычисляет определенный интеграл функции f(x) на отрезке [a, b] 
    методом трапеций с разбиением на n отрезков.
    """
    if n <= 0:
        raise ValueError("Количество разбиений n должно быть больше нуля.")
        
    h = (b - a) / n 
    total_sum = 0.5 * (f(a) + f(b))
    
    for i in range(1, n):
        x = a + i * h
        total_sum += f(x)
        
    return total_sum * h

In [5]:
%timeit integrate_trapezoidal_python(0.0, 10.0, 1000000)

83 ms ± 635 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [6]:
import numpy as np

def f_numpy(x):
    # Расчет идет сразу для всего массива точек x
    return x * x + 2.0 * x + 1.0

def integrate_trapezoidal_numpy(a, b, n):
    if n <= 0:
        raise ValueError("Количество разбиений n должно быть больше нуля.")
        
    h = (b - a) / n
    
    # array of n-1 inner points x (от 1 до n-1)
    # np.linspace создает сетку точек быстрее, чем цикл
    x_inner = np.linspace(a + h, b - h, n - 1)
    
    # Суммируем значения во всех внутренних точках и добавляем края
    total_sum = 0.5 * (f_numpy(a) + f_numpy(b)) + np.sum(f_numpy(x_inner))
    
    return total_sum * h


In [7]:
%timeit integrate_trapezoidal_numpy(0.0, 10.0, 1000000)

6.02 ms ± 122 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Сравнение производительности (Метод трапеций, N = 1 000 000)

| Реализация | Время выполнения | Относительное ускорение | Описание |
| :--- | :--- | :--- | :--- |
| **Чистый Python** | 84.7 ms ± 1.19 ms | Базовый уровень ($1\times$) | Медленный интерпретируемый цикл `for` |
| **NumPy (Векторизация)** | 5.61 ms ± 22.4 μs | В $\approx 15$ раз быстрее | Без циклов, вычисления на C под капотом NumPy |
| **Cython (Оптимизированный)** | **908 μs ± 5.3 μs** | **В $\approx 93$ раза быстрее** | Полная компиляция в машинный код и `inline`-функции |
